# Policy Gradient Methods: REINFORCE and Actor-Critic

## Overview

In previous labs, you learned **value-based methods** (Q-learning, DQN) that:
- Estimate Q(s, a) or V(s)
- Derive policy implicitly (e.g., ε-greedy, argmax Q)

**Policy gradient methods** take a different approach:
- Directly parameterize the policy: π(a|s; θ)
- Optimize policy parameters θ to maximize expected return
- Can handle stochastic policies naturally

## When to Use Policy Gradients?

- **Continuous action spaces**: Value-based methods struggle
- **Stochastic policies**: Directly model probability distributions
- **High-dimensional actions**: No need to compute max over all actions

## This Lab

You will implement two fundamental policy gradient algorithms:
1. **REINFORCE**: Monte Carlo policy gradient (Williams, 1992)
2. **Actor-Critic**: Hybrid method combining policy and value function

## Learning Outcomes

By the end of this lab, you will:
- Understand the policy gradient theorem
- Implement policy networks from scratch
- Experience the high variance problem in REINFORCE
- See how Actor-Critic reduces variance
- Build foundation for modern RL algorithms (A3C, PPO, SAC)

**Estimated Completion Time**: 2.5-3 hours

## Theoretical Background

### 1. Policy Gradient Theorem

**Objective**: Maximize expected cumulative reward

$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[G_t] = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_{t=0}^{T} \gamma^t r_t\right]$$

where τ = (s₀, a₀, r₀, s₁, ...) is a trajectory sampled from policy π_θ.

**The Problem**: How do we compute ∇_θ J(θ) when:
- J(θ) involves expectation over trajectories
- Trajectories depend on policy π_θ
- Environment dynamics are unknown

**The Solution: Policy Gradient Theorem**

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t\right]$$

**Key Insight**: We can estimate the gradient using **samples**!

1. Collect trajectories by running π_θ in environment
2. Compute returns G_t for each timestep
3. Compute gradient: ∇_θ log π_θ(a_t|s_t) * G_t
4. Update: θ ← θ + α * ∇_θ J(θ)

**Intuition**:
- If action a_t led to high return G_t, increase π_θ(a_t|s_t)
- If action a_t led to low return G_t, decrease π_θ(a_t|s_t)
- The log gradient ∇_θ log π_θ(a|s) points in direction to increase π_θ(a|s)

### 2. REINFORCE Algorithm

**REINFORCE** implements the policy gradient theorem directly:

1. Generate episode: s₀, a₀, r₀, ..., s_T, a_T, r_T
2. For each timestep t:
   - Compute return: G_t = Σ_{k=t}^{T} γ^{k-t} r_k
3. Compute loss: L = -Σ_t log π_θ(a_t|s_t) * G_t
4. Backpropagate: θ ← θ + α * ∇_θ L

**Problem**: High variance! G_t varies a lot between trajectories.

### 3. Actor-Critic: Reducing Variance

**The High Variance Problem**: REINFORCE uses **Monte Carlo returns** G_t:
- Must wait until episode ends
- G_t = r_t + γr_{t+1} + γ²r_{t+2} + ...
- High variance: small changes in actions cause big changes in G_t

**Example**: In CartPole, episode might end after 10 steps or 200 steps. G_t variance is huge!

**Solution: Use a Value Function**

Instead of full return G_t, use **TD learning**:
- Critic estimates V(s; w)
- TD error: δ_t = r_t + γV(s_{t+1}) - V(s_t)
- Use δ_t as "advantage estimate" for actor update

**Actor-Critic Architecture**: Two networks

1. **Actor**: Policy network π(a|s; θ)
   - Outputs action probabilities
   - Updated using policy gradient

2. **Critic**: Value network V(s; w)
   - Estimates state value
   - Updated using TD learning (you already know this from Q-learning lab!)

**Update Rules**:

**Critic Update** (familiar from Q-learning):
$$w \leftarrow w - \alpha_w \cdot \nabla_w [\delta_t^2]$$
$$\delta_t = r_t + \gamma V(s_{t+1}; w) - V(s_t; w)$$

**Actor Update** (policy gradient with advantage):
$$\theta \leftarrow \theta + \alpha_\theta \cdot \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot \delta_t$$

**Why It Works**:
- **Lower Variance**: δ_t is single-step TD error, not full return
- **Faster Learning**: Update after every step, not after full episode
- **Bootstrapping**: Critic provides bias (estimates future), but reduces variance

**Bias-Variance Trade-off**:
- **REINFORCE**: No bias (unbiased estimate), high variance
- **Actor-Critic**: Some bias (from V(s) approximation), much lower variance
- In practice, Actor-Critic works much better!

## Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch import Tensor

import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns; sns.set()

from typing import Tuple, List

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## Environment Setup

In [ ]:
# Create CartPole environment
env = gym.make('CartPole-v1', max_episode_steps=200)

# Get environment dimensions
state_dim = env.observation_space.shape[0]  # 4 for CartPole
action_dim = env.action_space.n             # 2 for CartPole (left, right)

print(f"State dimension: {state_dim}")
print(f"Action dimension: {action_dim}")
print(f"Max episode steps: 200")

## Task 1: Policy Network (Actor)

Implement a policy network that outputs action probabilities.

**Architecture**:
- Input: state (4 dimensions for CartPole)
- Hidden: 128 neurons with ReLU activation
- Output: action probabilities (2 actions) with softmax

**Important**: The output should be a probability distribution over actions that sums to 1.

In [ ]:
class PolicyNetwork(nn.Module):
    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int = 128):
        """
        Policy Network (Actor) that outputs action probabilities.
        
        Args:
            state_dim: Dimension of state space
            action_dim: Dimension of action space
            hidden_dim: Number of hidden units
        """
        super(PolicyNetwork, self).__init__()
        self.state_dim = state_dim
        self.action_dim = action_dim
        
        # TODO: Define network layers
        # Layer 1: Linear(state_dim, hidden_dim)
        # Layer 2: Linear(hidden_dim, action_dim)
        # self.fc1 = ...
        # self.fc2 = ...
        
    def forward(self, x: Tensor) -> Tensor:
        """
        Forward pass through the network.
        
        Args:
            x: State tensor of shape (batch_size, state_dim)
            
        Returns:
            Action probabilities of shape (batch_size, action_dim)
        """
        # TODO: Implement forward pass
        # 1. Apply first layer with ReLU activation
        # 2. Apply second layer with softmax activation
        # HINT: Use F.relu() and F.softmax()
        # HINT: softmax should be applied along dimension 1 (actions)
        # x = F.relu(self.fc1(x))
        # x = F.softmax(self.fc2(x), dim=1)
        # return x
        pass

## Task 2: Value Network (Critic)

Implement a value network that estimates V(s).

**Architecture**:
- Input: state (4 dimensions)
- Hidden: 128 neurons with ReLU activation
- Output: scalar value estimate (1 dimension)

**Important**: The output should be a single scalar value for each state.

In [ ]:
class ValueNetwork(nn.Module):
    def __init__(self, state_dim: int, hidden_dim: int = 128):
        """
        Value Network (Critic) that estimates state value V(s).
        
        Args:
            state_dim: Dimension of state space
            hidden_dim: Number of hidden units
        """
        super(ValueNetwork, self).__init__()
        self.state_dim = state_dim
        
        # TODO: Define network layers
        # Layer 1: Linear(state_dim, hidden_dim)
        # Layer 2: Linear(hidden_dim, 1) - output is scalar
        # self.fc1 = ...
        # self.fc2 = ...
        
    def forward(self, x: Tensor) -> Tensor:
        """
        Forward pass through the network.
        
        Args:
            x: State tensor of shape (batch_size, state_dim)
            
        Returns:
            Value estimates of shape (batch_size, 1)
        """
        # TODO: Implement forward pass
        # 1. Apply first layer with ReLU activation
        # 2. Apply second layer (no activation for value output)
        # x = F.relu(self.fc1(x))
        # x = self.fc2(x)
        # return x
        pass

## Helper Functions (Provided)

These functions are provided to help with action selection and return computation.

In [ ]:
def select_action(policy: nn.Module, state: np.ndarray, device: str = 'cpu') -> Tuple[int, Tensor]:
    """
    Sample action from policy distribution.
    
    Args:
        policy: Policy network
        state: Current state
        device: Device to run on
        
    Returns:
        action: Sampled action (integer)
        log_prob: Log probability of the action
    """
    state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
    probs = policy(state_tensor)
    
    # Create categorical distribution and sample
    action_dist = torch.distributions.Categorical(probs)
    action = action_dist.sample()
    log_prob = action_dist.log_prob(action)
    
    return action.item(), log_prob


def compute_returns(rewards: List[float], gamma: float = 0.99) -> Tensor:
    """
    Compute discounted returns for each timestep.
    
    Args:
        rewards: List of rewards from an episode
        gamma: Discount factor
        
    Returns:
        returns: Tensor of discounted returns
    """
    returns = []
    G = 0
    
    # Compute returns backwards from the end
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    
    return torch.FloatTensor(returns)

## Task 3: REINFORCE Training Loop

Implement the REINFORCE algorithm.

**Algorithm**:
1. Collect a full trajectory (episode)
2. Compute returns G_t for each timestep
3. Normalize returns (variance reduction)
4. Compute loss: -Σ log π(a_t|s_t) * G_t
5. Backpropagate and update policy

**Important**: 
- Returns should be normalized (subtract mean, divide by std) for stability
- Loss is negative because we want to maximize, not minimize

In [ ]:
def train_reinforce(
    env: gym.Env,
    policy: nn.Module,
    num_episodes: int = 1000,
    lr: float = 0.01,
    gamma: float = 0.99,
    device: str = 'cpu'
) -> List[float]:
    """
    Train agent using REINFORCE algorithm.
    
    Args:
        env: Gymnasium environment
        policy: Policy network
        num_episodes: Number of training episodes
        lr: Learning rate
        gamma: Discount factor
        device: Device to run on
        
    Returns:
        episode_rewards: List of total rewards per episode
    """
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    episode_rewards = []
    
    for episode in range(num_episodes):
        # Storage for episode data
        log_probs = []
        rewards = []
        
        # Collect trajectory
        state, _ = env.reset()
        done = False
        
        while not done:
            action, log_prob = select_action(policy, state, device)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            log_probs.append(log_prob)
            rewards.append(reward)
            
            state = next_state
        
        # TODO: Task 3 - Implement REINFORCE update
        
        # Step 1: Compute returns using the compute_returns function
        # returns = compute_returns(rewards, gamma)
        
        # Step 2: Normalize returns for variance reduction
        # HINT: returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        # The 1e-8 prevents division by zero
        # returns = ...
        
        # Step 3: Compute policy loss
        # FORMULA: L = -Σ log π(a_t|s_t) * G_t
        # HINT: Iterate through log_probs and returns, multiply them, and sum
        # HINT: loss should be negative of the sum (we want to maximize reward)
        # policy_loss = 0
        # for log_prob, G in zip(log_probs, returns):
        #     policy_loss = policy_loss + (-log_prob * G)
        
        # Step 4: Backpropagate and update
        # optimizer.zero_grad()
        # policy_loss.backward()
        # optimizer.step()
        
        # END TODO
        
        # Store episode reward
        episode_rewards.append(sum(rewards))
        
        # Logging
        if episode % 50 == 0:
            avg_reward = np.mean(episode_rewards[-50:])
            print(f"Episode {episode:4d} | Avg Reward: {avg_reward:6.2f}")
    
    return episode_rewards

## Task 4: Actor-Critic Training Loop

Implement the Actor-Critic algorithm.

**Algorithm**:
1. For each step in episode:
   - Select action from actor policy
   - Observe reward and next state
   - Compute TD error: δ = r + γV(s') - V(s)
   - Update critic: minimize δ²
   - Update actor: maximize log π(a|s) * δ

**Important**:
- TD error should be detached when used for actor update (prevent gradient flow to critic)
- If episode is done, next_value should be 0 (no future reward)

In [ ]:
def train_actor_critic(
    env: gym.Env,
    actor: nn.Module,
    critic: nn.Module,
    num_episodes: int = 1000,
    lr_actor: float = 0.001,
    lr_critic: float = 0.005,
    gamma: float = 0.99,
    device: str = 'cpu'
) -> List[float]:
    """
    Train agent using Actor-Critic algorithm.
    
    The Actor-Critic algorithm maintains two networks:
    - Actor (policy): π(a|s; θ) - selects actions
    - Critic (value): V(s; w) - estimates state value
    
    At each step:
    1. Actor selects action based on current policy
    2. Environment returns reward and next state
    3. Critic computes TD error: δ = r + γV(s') - V(s)
    4. Both networks update using δ
    
    Args:
        env: Gymnasium environment
        actor: Policy network
        critic: Value network
        num_episodes: Number of training episodes
        lr_actor: Learning rate for actor
        lr_critic: Learning rate for critic
        gamma: Discount factor
        device: Device to run on
        
    Returns:
        episode_rewards: List of total rewards per episode
    """
    actor_optimizer = optim.Adam(actor.parameters(), lr=lr_actor)
    critic_optimizer = optim.Adam(critic.parameters(), lr=lr_critic)
    episode_rewards = []
    
    for episode in range(num_episodes):
        state, _ = env.reset()
        done = False
        episode_reward = 0
        
        while not done:
            # Select action using actor
            action, log_prob = select_action(actor, state, device)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            episode_reward += reward
            
            # TODO: Task 4 - Implement Actor-Critic updates
            
            # Step 1: Prepare state tensors for networks
            # HINT: Networks expect batched input: (batch_size, state_dim)
            # Use .unsqueeze(0) to add batch dimension
            # state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
            # next_state_tensor = torch.FloatTensor(next_state).unsqueeze(0).to(device)
            
            # Step 2: Compute value estimates using critic
            # HINT: value should be a scalar - use .squeeze() to remove batch dimension
            # HINT: If episode is done, next_value = 0 (no future reward)
            # value = critic(state_tensor).squeeze()
            # next_value = 0 if done else critic(next_state_tensor).squeeze()
            
            # Step 3: Compute TD error (advantage estimate)
            # FORMULA: δ = r + γV(s') - V(s)
            # This is the "advantage" - how much better was this action than expected?
            # td_error = reward + gamma * next_value - value
            
            # Step 4: Update critic to minimize TD error squared
            # GOAL: Make V(s) accurate - reduce (δ)²
            # HINT: Use .pow(2) or td_error ** 2
            # critic_loss = td_error.pow(2)
            # critic_optimizer.zero_grad()
            # critic_loss.backward()
            # critic_optimizer.step()
            
            # Step 5: Update actor using policy gradient
            # GOAL: Increase probability of actions with positive advantage
            # FORMULA: L_actor = -log π(a|s) * δ
            # CRITICAL: Use .detach() on td_error to stop gradient flow to critic!
            # actor_loss = -log_prob * td_error.detach()
            # actor_optimizer.zero_grad()
            # actor_loss.backward()
            # actor_optimizer.step()
            
            # END TODO
            
            state = next_state
        
        # Store episode reward
        episode_rewards.append(episode_reward)
        
        # Logging
        if episode % 50 == 0:
            avg_reward = np.mean(episode_rewards[-50:])
            print(f"Episode {episode:4d} | Avg Reward: {avg_reward:6.2f}")
    
    return episode_rewards

## Task 5: Training and Comparison

Train both REINFORCE and Actor-Critic algorithms and compare their performance.

**What to observe**:
- Which algorithm converges faster?
- Which algorithm has more stable learning (less variance)?
- Do both reach similar final performance?

In [ ]:
# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# TODO: Task 5 - Train both algorithms

# Part 1: Train REINFORCE
print("\n" + "="*50)
print("Training REINFORCE...")
print("="*50)

# Initialize policy network for REINFORCE
# reinforce_policy = PolicyNetwork(state_dim, action_dim).to(device)

# Train REINFORCE
# reinforce_rewards = train_reinforce(
#     env=env,
#     policy=reinforce_policy,
#     num_episodes=1000,
#     lr=0.01,
#     gamma=0.99,
#     device=device
# )

# Part 2: Train Actor-Critic
print("\n" + "="*50)
print("Training Actor-Critic...")
print("="*50)

# Initialize actor and critic networks
# actor = PolicyNetwork(state_dim, action_dim).to(device)
# critic = ValueNetwork(state_dim).to(device)

# Train Actor-Critic
# ac_rewards = train_actor_critic(
#     env=env,
#     actor=actor,
#     critic=critic,
#     num_episodes=1000,
#     lr_actor=0.001,
#     lr_critic=0.005,
#     gamma=0.99,
#     device=device
# )

# END TODO

print("\nTraining complete!")

## Visualization (Provided)

Plot learning curves to compare REINFORCE and Actor-Critic.

In [ ]:
def moving_average_with_variance(data: List[float], window_size: int = 50) -> Tuple:
    """
    Calculate moving average and variance over the given window size.
    
    Args:
        data: List of values
        window_size: Size of moving window
        
    Returns:
        indices: X-axis indices
        means: Moving average values
        bounds: [lower_bounds, upper_bounds] for variance
    """
    if len(data) < window_size:
        return [], [], []
    
    indices = np.arange(window_size - 1, len(data))
    means = []
    upper_bounds = []
    lower_bounds = []
    
    for i in range(window_size - 1, len(data)):
        window = data[i - window_size + 1 : i + 1]
        mean_val = np.mean(window)
        std_val = np.std(window)
        means.append(mean_val)
        upper_bounds.append(mean_val + std_val)
        lower_bounds.append(mean_val - std_val)
    
    return indices, means, [lower_bounds, upper_bounds]


# Plot comparison
fig, ax = plt.subplots(figsize=(12, 6))

# Plot REINFORCE
x_rf, rf_mean, rf_var = moving_average_with_variance(reinforce_rewards, window_size=50)
ax.plot(x_rf, rf_mean, color='blue', label='REINFORCE', alpha=0.8, linewidth=2)
ax.fill_between(x_rf, rf_var[0], rf_var[1], alpha=0.2, color='blue')

# Plot Actor-Critic
x_ac, ac_mean, ac_var = moving_average_with_variance(ac_rewards, window_size=50)
ax.plot(x_ac, ac_mean, color='orange', label='Actor-Critic', alpha=0.8, linewidth=2)
ax.fill_between(x_ac, ac_var[0], ac_var[1], alpha=0.2, color='orange')

ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Average Reward (50-episode window)', fontsize=12)
ax.set_title('REINFORCE vs Actor-Critic on CartPole-v1', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print final statistics
print("\nFinal Performance (last 100 episodes):")
print(f"REINFORCE:    {np.mean(reinforce_rewards[-100:]):.2f} ± {np.std(reinforce_rewards[-100:]):.2f}")
print(f"Actor-Critic: {np.mean(ac_rewards[-100:]):.2f} ± {np.std(ac_rewards[-100:]):.2f}")

## Analysis Questions

After running the experiments, answer these questions:

### Question 1: Convergence Speed
Which algorithm reaches optimal performance (reward ~200) faster? Approximately how many episodes does each need?

**Your answer**: 

### Question 2: Stability
Which algorithm shows less variance in learning curves? Why do you think this is?

**Your answer**: 

### Question 3: Final Performance
Do both algorithms achieve similar final rewards? What is the average reward in the last 100 episodes for each?

**Your answer**: 

### Question 4: Variance Observation
Can you see the high variance problem in REINFORCE? How does it manifest in the learning curve?

**Your answer**: 

### Question 5: Advantage of Critic
How does the value function (critic) help Actor-Critic learn better than REINFORCE? Explain in terms of bias and variance.

**Your answer**: 

## Bonus Experiments (Optional)

Try these additional experiments to deepen your understanding:

### Experiment 1: Learning Rate Sensitivity
- Try different learning rates for actor and critic
- What happens if actor learns faster than critic?
- What's the optimal ratio?

### Experiment 2: Network Architecture
- Experiment with different hidden layer sizes (64, 128, 256)
- Add a second hidden layer
- How does this affect learning?

### Experiment 3: Entropy Regularization
- Add entropy bonus to actor loss to encourage exploration
- Formula: `actor_loss = -log_prob * td_error - entropy_coef * entropy`
- How does this affect exploration vs exploitation?

### Experiment 4: REINFORCE with Baseline
- Implement REINFORCE with a learned baseline (value function)
- Compare with vanilla REINFORCE
- This is actually a step toward Actor-Critic!

## Conclusion

### Key Takeaways

1. **Policy Gradient Theorem**: Enables direct policy optimization through gradient ascent
   - ∇_θ J(θ) = E[∇_θ log π(a|s) * G_t]
   - Can be estimated using samples from the environment

2. **REINFORCE**: Simple but suffers from high variance
   - Uses Monte Carlo returns (full episode)
   - Unbiased but high variance
   - Requires normalization for stability

3. **Actor-Critic**: Reduces variance using value function
   - Actor: policy network
   - Critic: value network (TD learning)
   - Lower variance, faster convergence
   - Some bias from value function approximation

4. **Bias-Variance Trade-off**:
   - REINFORCE: No bias, high variance
   - Actor-Critic: Small bias, much lower variance
   - In practice, Actor-Critic works better!

### Connections to Advanced Methods

The algorithms you implemented today are the foundation for modern deep RL:

- **A3C/A2C**: Actor-Critic + Parallel training
- **PPO**: Actor-Critic + Trust region constraints
- **SAC**: Actor-Critic + Maximum entropy + Off-policy
- **DDPG/TD3**: Continuous action spaces

### Further Reading

- Lilian Weng's Blog: [Policy Gradient Algorithms](https://lilianweng.github.io/posts/2018-04-08-policy-gradient/)
- Sutton & Barto, Chapter 13: "Policy Gradient Methods"
- OpenAI Spinning Up: [Introduction to RL](https://spinningup.openai.com/)

### Congratulations!

You've successfully implemented two fundamental policy gradient algorithms and understand the foundation of modern reinforcement learning!